In [8]:
import pandas as pd
import numpy as np


def _norm(df):
    """lower/strip column names so joins survive capitalisation drift."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    return df

In [9]:
# Merge fixations.csv, aoi_fixations.csv, and metadata document with Recording ID & Participant ID information

def build_fixation_table(fixations_path, aoi_path, metadata_path,
                         aoi_labels=None, verbose=True):
    fix = _norm(pd.read_csv(fixations_path))
    aoi = _norm(pd.read_csv(aoi_path))
    meta = _norm(pd.read_csv(metadata_path))

    # one-hot the AOI labels per (recording, fixation): multi-label safe
    oh = pd.crosstab([aoi["recording id"], aoi["fixation id"]], aoi["aoi label"])
    oh = (oh > 0).astype("int8")
    if aoi_labels is not None:
        oh = oh.reindex(columns=aoi_labels, fill_value=0)
    oh.columns = [f"AOI_{c}" for c in oh.columns]
    aoi_cols = list(oh.columns)
    oh = oh.reset_index()

    merged = fix.merge(oh, on=["recording id", "fixation id"], how="left")
    merged[aoi_cols] = merged[aoi_cols].fillna(0).astype("int8")

    merged = merged.merge(meta, on="recording id", how="left")   # full meta -> carries physio id etc.
    if verbose and merged["team id"].isna().any():
        print(f"WARNING: {merged['team id'].isna().sum()} rows missing Team ID.")

    rename = {"team id": "Team ID", "recording id": "Recording ID",
              "fixation id": "Fixation ID", "start timestamp [ns]": "Start",
              "end timestamp [ns]": "End", "duration [ms]": "Duration (ms)"}
    merged = merged.rename(columns=rename)

    base = ["Team ID", "Recording ID", "Fixation ID", "Start", "End", "Duration (ms)"]
    extra_meta = [c for c in merged.columns
                  if c not in base + aoi_cols + list(fix.columns)
                  and c not in ("recording id", "team id")]
    out = merged[base + aoi_cols + extra_meta].copy()
    out["Start"] = out["Start"].astype("int64")
    out["End"] = out["End"].astype("int64")
    return out, aoi_cols

In [ ]:
# Merge fixations_on_face.csv to add an AOI_Face column (1 = fixation on face, 0 = not)

def add_face_column(fix, aoi_cols, face_path):
    face = _norm(pd.read_csv(face_path))
    face = face[["recording id", "fixation id", "fixation on face"]].copy()
    face = face.rename(columns={"recording id": "Recording ID",
                                "fixation id": "Fixation ID",
                                "fixation on face": "AOI_Face"})
    face["AOI_Face"] = face["AOI_Face"].astype(bool).astype("int8")

    fix = fix.merge(face[["Recording ID", "Fixation ID", "AOI_Face"]],
                    on=["Recording ID", "Fixation ID"], how="left")
    fix["AOI_Face"] = fix["AOI_Face"].fillna(0).astype("int8")

    aoi_cols = aoi_cols + ["AOI_Face"]
    return fix, aoi_cols

In [10]:
# Reconstruct UNIX timestamp per block in the physiological file

def add_physio_unix(physio, marker_col="DataSyncMarker_channel_1",
                    device_col="timestamp", group_col=None, gap_warn_s=1.0,
                    unix_lo=1e9, unix_hi=2e9, verbose=True):
    def _one(g, label=""):
        g = g.sort_values(device_col).reset_index(drop=True)

        d = np.diff(g[device_col].to_numpy(float))
        if len(d) and d.max() > gap_warn_s:
            print(f"WARNING {label}device-clock gap {d.max():.1f}s -> may contain two recordings.")

        ch = pd.to_numeric(g[marker_col], errors="coerce")
        valid = ch.notna() & ch.between(unix_lo, unix_hi)     # only real UNIX-second markers
        is_start = valid & (ch != ch.shift())
        n_bad = int((g[marker_col].notna() & ~valid & (ch != ch.shift())).sum())
        if n_bad:
            print(f"NOTE {label}ignored {n_bad} non-UNIX marker value(s).")

        if not is_start.any():
            print(f"WARNING {label}no valid sync markers -> cannot reconstruct UNIX time.")
            g["unix_ns"] = pd.NA; g["dt_utc"] = pd.NaT
            return g

        a_ts = g.loc[is_start, device_col].to_numpy(float)
        a_off = ch[is_start].to_numpy(float) - a_ts
        order = np.argsort(a_ts)
        a_ts, a_off = a_ts[order], a_off[order]

        off = np.interp(g[device_col].to_numpy(float), a_ts, a_off)
        g["unix_ns"] = ((g[device_col].to_numpy(float) + off) * 1e9).round().astype("int64")
        g["dt_utc"] = pd.to_datetime(g["unix_ns"], unit="ns", utc=True)
        if verbose:
            print(f"{label}{len(a_off)} anchors | offset spread {(a_off.max()-a_off.min())*1000:.1f} ms")
        return g

    if group_col is None:
        return _one(physio)
    parts = [_one(g, label=f"ID {gid}: ") for gid, g in physio.groupby(group_col)]
    return pd.concat(parts, ignore_index=True)

In [11]:
# Check for fixation count to calculate fixation lost due to 40ms physio sampling constraint

def fixation_sample_counts(physio, fix, physio_group="ID", fix_group="Team ID", verbose=True):
    fix = fix.copy()
    fix["n_physio_samples"] = 0
    for tid, fg in fix.groupby(fix_group):
        ts = np.sort(physio.loc[physio[physio_group] == tid, "unix_ns"].to_numpy("int64"))
        if len(ts) == 0:
            continue
        lo = np.searchsorted(ts, fg["Start"].to_numpy("int64"), side="left")
        hi = np.searchsorted(ts, fg["End"].to_numpy("int64"), side="right")
        fix.loc[fg.index, "n_physio_samples"] = hi - lo
    if verbose:
        zero = int((fix["n_physio_samples"] == 0).sum())
        print(f"{zero}/{len(fix)} fixations ({100*zero/len(fix):.1f}%) contain 0 physio samples.")
        if "Duration (ms)" in fix:
            print(f"   median duration of those: {fix.loc[fix['n_physio_samples']==0,'Duration (ms)'].median()} ms")
    return fix

In [12]:
# Joining with physiological file

def annotate_physio(physio, fix, aoi_cols, physio_by="ID", fix_by="Team ID",
                    fix_id_col="Fixation ID"):
    physio = physio.sort_values("unix_ns").reset_index(drop=True)
    fcols = [fix_by, "Recording ID", fix_id_col, "Start", "End", "Duration (ms)"] + aoi_cols
    fix = fix.sort_values("Start")[fcols].reset_index(drop=True)

    j = pd.merge_asof(physio, fix,
                      left_on="unix_ns", right_on="Start",
                      left_by=physio_by, right_by=fix_by,   # keep the join within a participant
                      direction="backward")
    inside = (j["unix_ns"] <= j["End"]) & j["End"].notna()
    j["in_fixation"] = inside.astype("int8")
    for c in aoi_cols:
        j[c] = (j[c].fillna(0).astype("int8") * inside.astype("int8"))
    j.loc[~inside, ["Recording ID", fix_id_col, "Start", "End", "Duration (ms)"]] = np.nan
    return j

In [ ]:
# Execution Cell

base = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV"

fix, aoi_cols = build_fixation_table(
    f"{base}/fixations.csv",
    f"{base}/aoi_fixations.csv",
    f"{base}/Recording ID Metadata.csv",
    aoi_labels=["North", "South"],
)

fix, aoi_cols = add_face_column(fix, aoi_cols, f"{base}/fixations_on_face.csv")

physio = pd.read_csv(f"{base}/all_participants_combined.csv")
physio = add_physio_unix(physio, group_col="ID")          # per-participant clock anchoring
fix = fixation_sample_counts(physio, fix)                 # physio_group="ID", fix_group="Team ID"
merged = annotate_physio(physio, fix, aoi_cols)           # physio_by="ID", fix_by="Team ID"

merged[["ID", "unix_ns", "dt_utc", "Recording ID", "Fixation ID", "in_fixation"] + aoi_cols].head()

merged.to_csv(f"{base}/aoi_physio_merged.csv", index=False)